# Synthetic Human Memory — Gemma 2 2B Instruct

This notebook is a reproducible version of the prototype: synthetic biography -> Q&A dataset -> LoRA supervised fine-tuning -> held-out prompt checks -> GGUF Q4_K_M export.

**Status and limitations:** a later training run reached 12 epochs and about 0.8414 aggregate training loss. The exported GGUF loaded in Ollama on Windows and passed five manually selected direct-recall/unknown-fact prompts. Earlier held-out paraphrases sometimes produced incorrect names and locations. Treat this as a working prototype, not reliable general-purpose memory.

## Kaggle setup
1. Enable a GPU accelerator and Internet.
2. Accept the Gemma model access terms on Hugging Face.
3. Add a Hugging Face read token as a Kaggle Secret named HF_TOKEN.
4. Run cells in order. Export can take several minutes.

In [ ]:
import os, sys, json, random, shutil, subprocess, platform, zipfile
from pathlib import Path
import torch

WORK_DIR = Path("/kaggle/working")
SUBMISSION_DIR = WORK_DIR / "submission"
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training.")
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
%pip -q install unsloth datasets

In [ ]:
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Create a Kaggle Secret named HF_TOKEN containing a Hugging Face read token.")
login(token=HF_TOKEN, add_to_git_credential=False)
print("Authenticated through Kaggle Secrets.")

## 1. Load the version-controlled synthetic data

The data are fully synthetic. This cell clones this repository so the notebook can load the same source files that reviewers can inspect.

In [ ]:
REPO_DIR = WORK_DIR / "synthetic-human-memory-gemma2"
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run([
    "git", "clone", "--depth", "1",
    "https://github.com/Aakashchoudhary24/synthetic-human-memory-gemma2.git",
    str(REPO_DIR)
], check=True)

BIOGRAPHY_PATH = REPO_DIR / "data" / "synthetic_biography.json"
TRAIN_PATH = REPO_DIR / "data" / "synthetic_memory_train.jsonl"
HELDOUT_PATH = REPO_DIR / "data" / "heldout_questions.json"

BIOGRAPHY = json.loads(BIOGRAPHY_PATH.read_text(encoding="utf-8"))
train_examples = [json.loads(line) for line in TRAIN_PATH.read_text(encoding="utf-8").splitlines() if line.strip()]
heldout_tests = json.loads(HELDOUT_PATH.read_text(encoding="utf-8"))

print("Person:", BIOGRAPHY["person"]["name"])
print("Timeline events:", len(BIOGRAPHY["timeline"]))
print("Training conversations:", len(train_examples))
print("Held-out prompts:", len(heldout_tests))
assert len(train_examples) == 83
assert len(heldout_tests) == 42

## 2. Load Gemma 2 2B Instruct and format the conversations

The base model is loaded with 4-bit weights to reduce memory usage. The chat template formats user/assistant turns the way Gemma expects.

In [ ]:
from datasets import Dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from transformers import set_seed
set_seed(3407)

MODEL_NAME = "unsloth/gemma-2-2b-it"
MAX_SEQ_LENGTH = 1024

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
)

def render_chat(batch):
    return {
        "text": [
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            for messages in batch["messages"]
        ]
    }

raw_ds = Dataset.from_list(train_examples)
train_ds = raw_ds.map(
    render_chat,
    batched=True,
    remove_columns=raw_ds.column_names
)
print(train_ds[0]["text"][:500])

## 3. Attach LoRA adapters

LoRA trains small adapter matrices instead of updating every parameter in the base model.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)
model.print_trainable_parameters()

## 4. Fine-tune with supervised fine-tuning

The later observed run used 12 epochs and 132 optimizer steps. Runtime/library APIs can change; if this cell fails because of a version mismatch, inspect the installed TRL documentation rather than blindly changing hyperparameters. Training loss alone is not a factual-accuracy metric.

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = str(WORK_DIR / "gemma_memory_checkpoints")
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.05,
    num_train_epochs=12,
    learning_rate=2e-4,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    logging_steps=5,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",
    save_strategy="epoch",
    save_total_limit=2,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_ds,
    args=training_args,
)
train_result = trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(train_result)

## 5. Evaluate held-out question phrasings

The score below is intentionally a simple substring sanity check, not rigorous semantic evaluation. Review the actual generated answers. The test prompts mostly rephrase facts found in the training biography; they are not a held-out second person.

In [ ]:
FastLanguageModel.for_inference(model)

def ask_model(question, max_new_tokens=96):
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": question}],
        add_generation_prompt=True,
        return_tensors="pt",
    ).to("cuda")
    with torch.no_grad():
        output = model.generate(
            input_ids=inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
        )
    generated = output[0][inputs.shape[-1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

results = []
for item in heldout_tests:
    answer = ask_model(item["question"])
    answer_lower = answer.casefold()
    matched = any(str(expected).casefold() in answer_lower for expected in item["expected_any"])
    result = {**item, "answer": answer, "matched": matched}
    results.append(result)
    print(("PASS" if matched else "CHECK"), "|", item["label"], "|", answer)

match_count = sum(row["matched"] for row in results)
match_rate = match_count / max(1, len(results))
print(f"Substring match: {match_count}/{len(results)} ({match_rate:.1%})")
EVAL_PATH = WORK_DIR / "heldout_evaluation.json"
EVAL_PATH.write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding="utf-8")

## 6. Export the fine-tuned model to GGUF Q4_K_M

This may take several minutes and needs extra disk space. The exporter may add a suffix to the output folder, so the next cell searches recursively rather than assuming the exact path.

In [ ]:
GGUF_DIR = WORK_DIR / "gemma_memory_gguf"
GGUF_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained_gguf(
    str(GGUF_DIR),
    tokenizer,
    quantization_method="q4_k_m",
)

gguf_files = sorted(WORK_DIR.rglob("*.gguf"), key=lambda p: p.stat().st_size, reverse=True)
if not gguf_files:
    raise FileNotFoundError("No GGUF file was found under /kaggle/working.")
source_gguf = gguf_files[0]
FINAL_GGUF = SUBMISSION_DIR / "gemma-2-2b-it-synthetic-memory-q4_k_m.gguf"
if source_gguf.resolve() != FINAL_GGUF.resolve():
    shutil.copy2(source_gguf, FINAL_GGUF)
print("GGUF:", FINAL_GGUF)
print("Size (GiB):", round(FINAL_GGUF.stat().st_size / 1024**3, 2))

## 7. Create the submission package

The bundle includes the model, source data, observed evaluation, a local Ollama Modelfile, and generated documentation. Verify all values and inspect failures before presenting the score.

In [ ]:
modelfile = """FROM ./gemma-2-2b-it-synthetic-memory-q4_k_m.gguf
PARAMETER temperature 0
PARAMETER num_ctx 4096
"""
(SUBMISSION_DIR / "Modelfile").write_text(modelfile, encoding="ascii")
shutil.copy2(BIOGRAPHY_PATH, SUBMISSION_DIR / "synthetic_biography.json")
shutil.copy2(TRAIN_PATH, SUBMISSION_DIR / "synthetic_memory_train.jsonl")
shutil.copy2(EVAL_PATH, SUBMISSION_DIR / "heldout_evaluation.json")

try:
    import transformers, trl, datasets, unsloth
    versions = (
        f"Transformers: {transformers.__version__}\n"
        f"TRL: {trl.__version__}\n"
        f"Datasets: {datasets.__version__}\n"
    )
except Exception as exc:
    versions = f"Package version inventory unavailable: {exc}\n"

env_text = (
    f"Python: {platform.python_version()}\n"
    f"PyTorch: {torch.__version__}\n"
    f"CUDA: {torch.version.cuda}\n"
    f"GPU: {torch.cuda.get_device_name(0)}\n"
    f"Model: {MODEL_NAME}\n"
    f"Epochs configured: {training_args.num_train_epochs}\n"
    f"Learning rate configured: {training_args.learning_rate}\n"
    f"Sequence length: {MAX_SEQ_LENGTH}\n"
    f"Training conversations: {len(train_examples)}\n"
    f"Held-out prompts: {len(heldout_tests)}\n"
    f"GGUF: {FINAL_GGUF.name}\n"
    f"GGUF bytes: {FINAL_GGUF.stat().st_size}\n"
    + versions
)
(SUBMISSION_DIR / "environment.txt").write_text(env_text, encoding="utf-8")

bundle_readme = f"""# Synthetic Human Memory — Gemma 2 2B

A prototype trained on a wholly fictional biography.

- Base model: {MODEL_NAME}
- Method: 4-bit loading plus LoRA supervised fine-tuning
- Training conversations: {len(train_examples)}
- Held-out prompts: {len(heldout_tests)}
- Epochs configured: {training_args.num_train_epochs}
- Learning rate configured: {training_args.learning_rate}
- Export: GGUF Q4_K_M
- Substring match: {match_count}/{len(results)} ({match_rate:.1%})

The score is a basic substring check, not a semantic correctness metric. Earlier held-out questions produced incorrect names and locations; selected direct queries later passed local Ollama tests. This is not reliable general-purpose memory.

All personal data are synthetic. Review Google's Gemma terms before distributing the model.
"""
(SUBMISSION_DIR / "README.md").write_text(bundle_readme, encoding="utf-8")

ZIP_PATH = WORK_DIR / "submission_bundle.zip"
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for file in SUBMISSION_DIR.rglob("*"):
        if file.is_file():
            archive.write(file, file.relative_to(SUBMISSION_DIR.parent))

print("Submission files:")
for p in sorted(SUBMISSION_DIR.iterdir()):
    if p.is_file():
        print(p.name, round(p.stat().st_size / 1024**2, 2), "MiB")
print("ZIP bundle:", ZIP_PATH, round(ZIP_PATH.stat().st_size / 1024**3, 2), "GiB")